# 020 — Train Aerial Models

Extracts training pixels from 512×512 tiles and trains RF or XGBoost model.
Configure MODEL_TYPE, USE_FILTER, USE_SUBSET and USE_NIR_CONTRAST in config.py before running.

Run after `010_prepare_aerial.ipynb`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from config import *
from helpers import *
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              accuracy_score, confusion_matrix)
from sklearn.inspection import permutation_importance

print(f"BASE_DIR:     {BASE_DIR}")
print(f"Cores:        {N_CORES}")
print(f"Model type:   {MODEL_TYPE}")
print(f"Filter:       {USE_FILTER}")
print(f"NIR contrast: {USE_NIR_CONTRAST}")
print(f"Subset:       {USE_SUBSET}")
print(f"NDVI filter:  {NDVI_MIN} – {NDVI_MAX}")
check_required_files()

In [ ]:
# ── Step 1: load tile pairs ───────────────────────────────────────────────────
print("── Step 1: Load tile pairs ─────────────────────────────────")

if USE_SUBSET:
    df_subset    = pd.read_csv(SUBSET_PATH)
    subset_files = df_subset["file"].tolist()
    print(f"Mode: subset ({len(subset_files)} tiles)")
else:
    subset_files = [f for f in os.listdir(TILES_IMG_DIR) if f.endswith(".tif")]
    print(f"Mode: all tiles ({len(subset_files)} tiles)")

pairs = [
    (os.path.join(TILES_IMG_DIR, f), os.path.join(TILES_MASK_DIR, f))
    for f in sorted(subset_files)
    if os.path.exists(os.path.join(TILES_IMG_DIR, f))
    and os.path.exists(os.path.join(TILES_MASK_DIR, f))
]
print(f"Valid pairs: {len(pairs)}")

In [ ]:
# ── Step 2: feature extraction ────────────────────────────────────────────────
FEATURE_NAMES = ["R", "G", "B", "NIR", "NDVI", "NDWI", "EVI"]
if USE_NIR_CONTRAST:
    # FEATURE_NAMES += ["NIR_contrast_5"]  # uncomment to add 5px
    FEATURE_NAMES += ["NIR_contrast_25"]
NDVI_IDX = FEATURE_NAMES.index("NDVI")

print(f"Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}")

def run_extraction(pairs, use_filter, cache_path):
    """
    Extract features and labels from tile pairs.
    use_filter=True:  only lupine pixels with NDVI in [NDVI_MIN, NDVI_MAX]
    use_filter=False: all pixels, no label filtering
    Returns X (n_pixels, n_features), y (n_pixels,)
    """
    from tqdm import tqdm

    if os.path.exists(cache_path):
        with open(cache_path) as f:
            cache = {k: np.array(v) for k, v in json.load(f).items()}
        print(f"Loaded cache: {len(cache)} tiles ({os.path.basename(cache_path)})")
    else:
        cache = {}
        print(f"No cache — will create: {os.path.basename(cache_path)}")

    X_list, y_list = [], []
    cache_updated  = False
    skipped        = 0
    total_lupine   = 0
    total_bg       = 0

    label = "filtered" if use_filter else "unfiltered"
    pbar  = tqdm(pairs, desc=f"  Extracting ({label})", unit="tile")

    for img_path, mask_path in pbar:
        tile_name = os.path.basename(img_path)

        with rasterio.open(img_path)  as src: img  = src.read().astype(np.float32)
        with rasterio.open(mask_path) as src: mask = src.read(1).astype(np.uint8)

        N         = img.shape[1] * img.shape[2]
        features  = engineer_features(img, use_nir_contrast=USE_NIR_CONTRAST)
        mask_flat = mask.reshape(-1)

        if use_filter:
            ndvi             = features[:, NDVI_IDX]
            plausible_lupine = (mask_flat == 1) & (ndvi >= NDVI_MIN) & (ndvi <= NDVI_MAX)
            background       = (mask_flat == 0)
            clean_mask       = np.where(plausible_lupine, 1, np.where(background, 0, 255))
            sample_pool      = np.where(clean_mask != 255)[0]
            labels           = clean_mask
        else:
            sample_pool = np.arange(N)
            labels      = mask_flat

        if len(sample_pool) == 0:
            skipped += 1; continue

        if tile_name in cache:
            idx = cache[tile_name]
            if use_filter:
                idx = idx[np.isin(idx, sample_pool)]
        else:
            rng   = np.random.default_rng(seed=abs(hash(tile_name)) % (2**32))
            n_smp = min(SAMPLES_PER_TILE, len(sample_pool))
            idx   = rng.choice(sample_pool, n_smp, replace=False)
            cache[tile_name] = idx.tolist()
            cache_updated    = True

        if len(idx) == 0:
            skipped += 1; continue

        X_list.append(features[idx])
        y_list.append(labels[idx])

        total_lupine += int((labels[idx] == 1).sum())
        total_bg     += int((labels[idx] == 0).sum())

        pbar.set_postfix({
            "lupine": f"{total_lupine:,}",
            "bg":     f"{total_bg:,}",
            "skip":   skipped,
        })

    if cache_updated:
        with open(cache_path, "w") as f:
            json.dump({k: v.tolist() if hasattr(v, 'tolist') else v
                       for k, v in cache.items()}, f)
        print(f"Cache saved: {os.path.basename(cache_path)}")

    X = np.vstack(X_list)
    y = np.hstack(y_list)
    print(f"X shape: {X.shape}  Skipped: {skipped}")
    print(f"Lupine: {(y==1).sum():,} ({(y==1).mean():.1%})  BG: {(y==0).sum():,}")
    return X, y

print("Extraction function ready ✓")

In [ ]:
# ── Step 3: run extraction ────────────────────────────────────────────────────
print("── Step 3: Extract pixels ──────────────────────────────────")

# Always extract both so we have the shared val set for evaluation
X_filt,   y_filt   = run_extraction(pairs, use_filter=True,  cache_path=CACHE_FILTERED)
X_unfilt, y_unfilt = run_extraction(pairs, use_filter=False, cache_path=CACHE_UNFILTERED)

# Select training data based on USE_FILTER
X_train_data = X_filt   if USE_FILTER else X_unfilt
y_train_data = y_filt   if USE_FILTER else y_unfilt
print(f"\nUsing {'filtered' if USE_FILTER else 'unfiltered'} data for training")

In [ ]:
# ── Step 4: train model ───────────────────────────────────────────────────────
print("── Step 4: Train model ─────────────────────────────────────")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_data, y_train_data,
    test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"  Train: {len(X_tr):,}  Val: {len(X_val):,}")

if MODEL_TYPE == "rf":
    model = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH,
        n_jobs=-1, random_state=RANDOM_STATE, verbose=0)
elif MODEL_TYPE == "xgb":
    from xgboost import XGBClassifier
    model = XGBClassifier(
        n_estimators=XGB_N_ESTIMATORS, max_depth=XGB_MAX_DEPTH,
        learning_rate=XGB_LR, n_jobs=-1, random_state=RANDOM_STATE,
        eval_metric="logloss", verbosity=0)
else:
    raise ValueError(f"Unknown MODEL_TYPE: {MODEL_TYPE} — must be 'rf' or 'xgb'")

model.fit(X_tr, y_tr)

# Build model name
data_label     = "filter"    if USE_FILTER      else "nofilter"
texture_label  = "_nir_5_25"    if USE_NIR_CONTRAST else "_notexture"
subset_label   = "_subset"   if USE_SUBSET       else "_all"
MODEL_NAME_OUT = f"{MODEL_TYPE}_{data_label}{texture_label}{subset_label}"

print(f"  Model: {MODEL_NAME_OUT} ✓")

In [ ]:
# ── Step 5: evaluate ─────────────────────────────────────────────────────────
print("── Step 5: Evaluate ────────────────────────────────────────")

# Own val set
pred_val = model.predict(X_val)
print(f"  Own val set ({len(X_val):,} px):")
print(f"    F1:        {f1_score(y_val, pred_val):.4f}")
print(f"    Precision: {precision_score(y_val, pred_val):.4f}")
print(f"    Recall:    {recall_score(y_val, pred_val):.4f}")
print(f"    Accuracy:  {accuracy_score(y_val, pred_val):.4f}")

# Shared unfiltered val set
_, X_val_shared, _, y_val_shared = train_test_split(
    X_unfilt, y_unfilt, test_size=0.2, random_state=42)
pred_shared = model.predict(X_val_shared)
print(f"\n  Shared unfiltered val set ({len(X_val_shared):,} px):")
print(f"    F1:        {f1_score(y_val_shared, pred_shared):.4f}")
print(f"    Precision: {precision_score(y_val_shared, pred_shared):.4f}")
print(f"    Recall:    {recall_score(y_val_shared, pred_shared):.4f}")
print(f"    Accuracy:  {accuracy_score(y_val_shared, pred_shared):.4f}")

# Save to results CSV
row = {
    "model":        MODEL_NAME_OUT,
    "f1":           round(f1_score(y_val_shared, pred_shared), 4),
    "precision":    round(precision_score(y_val_shared, pred_shared), 4),
    "recall":       round(recall_score(y_val_shared, pred_shared), 4),
    "accuracy":     round(accuracy_score(y_val_shared, pred_shared), 4),
    "model_type":   MODEL_TYPE,
    "use_filter":   USE_FILTER,
    "n_estimators": model.n_estimators,
    "n_train":      len(X_tr),
    "n_val":        len(X_val_shared),
    "eval_set":     "shared_unfiltered",
}
if os.path.exists(RESULTS_PATH):
    df_existing = pd.read_csv(RESULTS_PATH)
    df_existing = df_existing[df_existing["model"] != MODEL_NAME_OUT]
    df_results  = pd.concat([df_existing, pd.DataFrame([row])], ignore_index=True)
else:
    df_results = pd.DataFrame([row])
df_results.to_csv(RESULTS_PATH, index=False)
print(f"\n  Results saved → {RESULTS_PATH}")

In [ ]:
# ── Step 6: confusion matrix ──────────────────────────────────────────────────
print("── Step 6: Confusion matrix ────────────────────────────────")

QA_DIR = os.path.join(BASE_DIR, "qa_visuals")
os.makedirs(QA_DIR, exist_ok=True)

cm   = confusion_matrix(y_val_shared, pred_shared)
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"{MODEL_NAME_OUT} — shared unfiltered val set", fontsize=11)
sns.heatmap(cm,   annot=True, fmt=",",    cmap="Blues",
            xticklabels=["BG","Lupine"], yticklabels=["BG","Lupine"], ax=axes[0])
axes[0].set_title("Raw counts")
sns.heatmap(cm_n, annot=True, fmt=".3f",  cmap="Blues",
            xticklabels=["BG","Lupine"], yticklabels=["BG","Lupine"], ax=axes[1])
axes[1].set_title("Normalised")
plt.tight_layout()
plt.savefig(os.path.join(QA_DIR, f"confusion_{MODEL_NAME_OUT}.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Step 7: save model ────────────────────────────────────────────────────────
print("── Step 7: Save model ──────────────────────────────────────")

save_path = os.path.join(MODEL_DIR, f"{MODEL_NAME_OUT}_model.joblib")
if os.path.exists(save_path):
    print(f"  Already exists — overwriting: {MODEL_NAME_OUT}")
joblib.dump(model, save_path)
print(f"  Saved → {save_path}")
print(f"\n  Update MODEL_NAME in config.py to: {MODEL_NAME_OUT}")

In [ ]:
# ── Step 8: feature importance ────────────────────────────────────────────────
print("── Step 8: Feature importance ──────────────────────────────")

importances = model.feature_importances_
indices     = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 4))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [FEATURE_NAMES[i] for i in indices], rotation=45)
plt.title(f"Feature importances — {MODEL_NAME_OUT}")
plt.tight_layout()
plt.savefig(os.path.join(QA_DIR, f"feature_importance_{MODEL_NAME_OUT}.png"), dpi=150, bbox_inches="tight")
plt.show()

for i in indices:
    print(f"{FEATURE_NAMES[i]:20s} {importances[i]:.4f}")

In [ ]:
# ── Step 9: permutation importance ───────────────────────────────────────────
print("── Step 9: Permutation importance ─────────────────────────")

MAX_ROWS = 50_000
if len(X_val) > MAX_ROWS:
    idx_p  = np.random.default_rng(42).choice(len(X_val), MAX_ROWS, replace=False)
    X_p, y_p = X_val[idx_p], y_val[idx_p]
else:
    X_p, y_p = X_val, y_val

perm = permutation_importance(model, X_p, y_p, n_repeats=5, random_state=42, n_jobs=-1)
print(f"Permutation importance — {MODEL_NAME_OUT}")
for i in np.argsort(perm.importances_mean)[::-1]:
    print(f"{FEATURE_NAMES[i]:20s} {perm.importances_mean[i]:.4f} ± {perm.importances_std[i]:.4f}")

In [ ]:
# ── Step 10: visual QA ────────────────────────────────────────────────────────
print("── Step 10: Visual QA ──────────────────────────────────────")

import json as _json

def get_fixed_tiles(key, candidates, n=10, seed=42):
    cache = _json.load(open(FIXED_TILES_PATH)) if os.path.exists(FIXED_TILES_PATH) else {}
    if key not in cache:
        cache[key] = random.Random(seed).sample(candidates, min(n, len(candidates)))
        _json.dump(cache, open(FIXED_TILES_PATH, "w"), indent=2)
        print(f"Fixed '{key}' tiles saved")
    else:
        print(f"Loaded fixed '{key}' tiles")
    return cache[key]

def visualise_and_save(samples, model, model_name, split_label):
    save_dir = os.path.join(QA_DIR, model_name)
    os.makedirs(save_dir, exist_ok=True)
    for name in samples:
        img_path  = os.path.join(TILES_IMG_DIR, name)
        mask_path = os.path.join(TILES_MASK_DIR, name)
        if not os.path.exists(img_path) or not os.path.exists(mask_path): continue
        with rasterio.open(img_path)  as src: img  = src.read().astype(np.float32)
        with rasterio.open(mask_path) as src: mask = src.read(1).astype(np.uint8)
        features  = engineer_features(img, use_nir_contrast=USE_NIR_CONTRAST)
        C, H, W   = img.shape
        pred_mask = model.predict(features).reshape(H, W)
        rgb       = img[:3].transpose(1,2,0)
        rgb       = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(rgb);                    axes[0].set_title("RGB")
        axes[1].imshow(mask, cmap="gray");      axes[1].set_title(f"GT {mask.mean():.1%}")
        axes[2].imshow(pred_mask, cmap="gray"); axes[2].set_title(f"Pred {pred_mask.mean():.1%}")
        axes[3].imshow(rgb)
        axes[3].imshow(pred_mask, cmap="Reds", alpha=0.4); axes[3].set_title("Overlay")
        for ax in axes: ax.axis("off")
        fig.suptitle(f"{model_name} — {split_label} — {name}", fontsize=9)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{os.path.splitext(name)[0]}_{split_label}.png"),
                    dpi=150, bbox_inches="tight")
        plt.close()
    print(f"  Saved {len(samples)} QA images → {save_dir}")

all_images = sorted(f for f in os.listdir(TILES_IMG_DIR) if f.endswith(".tif"))

if USE_SUBSET:
    training_set = set(subset_files)
    training     = [f for f in all_images if f in training_set]
    unseen       = [f for f in all_images if f not in training_set]
    fixed_train  = get_fixed_tiles("train",  training, n=10)
    visualise_and_save(fixed_train, model, MODEL_NAME_OUT, "train")
    if unseen:
        fixed_unseen = get_fixed_tiles("unseen", unseen, n=10)
        visualise_and_save(fixed_unseen, model, MODEL_NAME_OUT, "unseen")
    else:
        print("  No unseen tiles — skipping unseen QA")
else:
    training = all_images
    fixed_train = get_fixed_tiles("train_all", training, n=10)
    visualise_and_save(fixed_train, model, MODEL_NAME_OUT, "train")
    print("  Skipping unseen QA — all tiles used for training")